In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from lightgbm import LGBMClassifier
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss,accuracy_score,balanced_accuracy_score
os.chdir("/home/pgcp-ai/MachineLearning/Cases/Wisconsin/")

In [2]:
df = pd.read_csv("BreastCancer.csv",index_col="Code")
df

,Clump,UniCell_Size,Uni_CellShape,MargAdh,SEpith,BareN,BChromatin,NoemN,Mitoses,Class
Code,,,,,,,,,,
61634,5,4,3,1,2,2,2,3,1,Benign
63375,9,1,2,6,4,10,7,7,2,Malignant
76389,10,4,7,2,2,8,6,1,1,Malignant
95719,6,10,10,10,8,10,7,10,7,Malignant
128059,1,1,1,1,2,5,5,1,1,Benign
...,...,...,...,...,...,...,...,...,...,...
1369821,10,10,10,10,5,10,10,10,7,Malignant
1371026,5,10,10,10,4,10,5,6,3,Malignant
1371920,5,1,1,1,2,1,3,2,1,Benign


In [3]:
le = LabelEncoder()

In [4]:
df['Class'] = le.fit_transform(df['Class'])

In [5]:
X,y = df.drop('Class',axis=1), df['Class']

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

In [10]:
gbm = LGBMClassifier(random_state=26, verbose = -1)

In [11]:
gbm.fit(X_train,y_train)
y_pred = gbm.predict(X_test)
y_pred_prob = gbm.predict_proba(X_test)
print(f"Log Loss: {log_loss(y_test,y_pred_prob)} \nAccuracy Score : {accuracy_score(y_test,y_pred)}")

Log Loss: 0.13712274268960134 
Accuracy Score : 0.9666666666666667


In [13]:
n_estimators = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []

for n in tqdm(n_estimators):
    for r in rate:
        for d in depth:
            gbm = LGBMClassifier(random_state=26, n_estimators=n, learning_rate=r, max_depth=d, verbose = -1)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            y_prob = gbm.predict_proba(X_test)
            scores.append([n, r, d, log_loss(y_test, y_pred_prob), accuracy_score(y_test, y_pred)])
            
df_scores = pd.DataFrame(scores, columns = ["Number Of Estimators", "Learning Rate", "Depth", "Log Loss", "Accuracy Score"])
df_scores.sort_values(["Log Loss", "Accuracy Score"], ascending = [True, False])

100%|████████████████| 5/5 [00:11<00:00,  2.32s/it]


,Number Of Estimators,Learning Rate,Depth,Log Loss,Accuracy Score
113,10,0.682783,5,0.137123,0.971429
266,25,0.713814,3,0.137123,0.971429
282,25,0.806907,4,0.137123,0.971429
399,50,0.589690,6,0.137123,0.971429
408,50,0.651752,5,0.137123,0.971429
...,...,...,...,...,...
600,100,0.000100,2,0.137123,0.657143
601,100,0.000100,3,0.137123,0.657143
602,100,0.000100,4,0.137123,0.657143
603,100,0.000100,5,0.137123,0.657143
